In [79]:
!pip install segmentation_models_pytorch --quiet

In [80]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vit_b_16, ViT_B_16_Weights
import segmentation_models_pytorch as smp

import numpy as np
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

def load_model(path='model.pth'):
    model = torch.load(path, weights_only=False)
    model.eval()

    return model

resize = (256, 256)
threshold = 0.5
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
LABEL_MAPS = {0: "Normal", 1: "Covid", 2: "Pneumonia"}

In [81]:
# Models

class CNNXRayNet(nn.Module):
    def __init__(self, n_classes_cls=3, n_classes_seg=3, dropout=0.3):
        super().__init__()

        # --- Shared CNN backbone ---
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # --- Project global feature to spatial map for UNet ---
        self.global_to_map = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 1),
            nn.ReLU(inplace=True),
        )

        # --- UNet for segmentation ---
        self.unet = smp.Unet(
            encoder_name="resnet34",      # lightweight encoder for speed
            encoder_weights=None,
            in_channels=64 + 1,           # backbone + global map
            classes=n_classes_seg,
        )

        # --- Classification head: global pooled features ---
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes_cls),
        )

    def forward(self, x):
        B, _, H, W = x.shape

        # Shared backbone
        feat = self.backbone(x)  # (B, 64, H, W)

        # Classification: global pooled
        cls_logits = self.classifier(feat)

        # Segmentation: inject global feature map
        global_map = self.global_to_map(feat).view(B, 1, 1, 1).expand(-1, 1, H, W)
        seg_input = torch.cat([feat, global_map], dim=1)
        seg_logits = self.unet(seg_input)

        return cls_logits, seg_logits


class InterconnectedXRayNet(nn.Module):
    def __init__(self, n_classes_cls=3, n_classes_seg=3, dropout=0.3):
        super().__init__()

        # --- Shared stem: 1->3 channels ---
        self.stem = nn.Sequential(
            nn.Conv2d(1, 3, 3, padding=1, bias=False),
            nn.BatchNorm2d(3),
            nn.ReLU(inplace=True),
        )

        # --- Vision Transformer ---
        weights = ViT_B_16_Weights.DEFAULT
        self.vit = vit_b_16(weights=weights)
        self.vit.heads = nn.Identity()  # remove classifier
        self.vit_feat_dim = self.vit.hidden_dim  # 768

        # --- CNN refinement block AFTER ViT patches ---
        self.cnn = nn.Sequential(
            nn.Conv2d(self.vit_feat_dim, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))  # pool to vector
        )
        self.cnn_out_dim = 256

        # --- Classification head ---
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.cnn_out_dim),
            nn.Linear(self.cnn_out_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes_cls),
        )

        # --- Project ViT CLS to spatial map for UNet ---
        self.vit_to_map = nn.Sequential(
            nn.Linear(self.vit_feat_dim, 1),
            nn.ReLU(inplace=True),
        )

        # --- UNet for segmentation ---
        self.unet = smp.Unet(
            encoder_name="resnet34",
            encoder_weights=None,
            in_channels=4,  # 3 stem + 1 ViT map
            classes=n_classes_seg,
        )

    def forward(self, x):
        B, _, H, W = x.shape

        # Shared stem
        stem = self.stem(x)

        # Resize for ViT
        vit_in = F.interpolate(stem, size=(224, 224), mode="bilinear", align_corners=False)

        # --- Extract embeddings from torchvision ViT ---
        x = self.vit._process_input(vit_in)              # (B, 196, 768) patch embeddings
        cls_token = self.vit.class_token.expand(B, -1, -1)  # (B, 1, 768)
        x = torch.cat((cls_token, x), dim=1)             # (B, 197, 768)
        x = x + self.vit.encoder.pos_embedding           # add positional embeddings
        x = self.vit.encoder.dropout(x)
        vit_features = self.vit.encoder.layers(x)        # (B, 197, 768)

        # Split CLS and patches
        vit_cls = vit_features[:, 0]        # (B, 768)
        vit_patches = vit_features[:, 1:]   # (B, 196, 768)

        # Reshape patches -> (B, 768, 14, 14)
        vit_patches = vit_patches.transpose(1, 2).reshape(B, self.vit_feat_dim, 14, 14)

        # CNN refinement on patch grid
        cnn_out = self.cnn(vit_patches).view(B, -1)  # (B, 256)
        cls_logits = self.classifier(cnn_out)

        # ViT CLS -> map for UNet
        vit_map = self.vit_to_map(vit_cls).view(B, 1, 1, 1).expand(-1, 1, H, W)
        seg_input = torch.cat([stem, vit_map], dim=1)
        seg_logits = self.unet(seg_input)

        return cls_logits, seg_logits

In [82]:
model_path_1 = "/kaggle/input/covid/pytorch/default/1/CNN Unet.pth"
model_path_2 = "/kaggle/input/covid/pytorch/default/1/CNN  ViT  Unet.pth"

model_novit = load_model(model_path_1).to(device)
model_vit = load_model(model_path_2).to(device)

In [83]:
# --- Preprocessing pipeline ---
def get_transform(resize=(256, 256)):
    return A.Compose([
        A.LongestMaxSize(max_size=resize[0]),   # keep aspect ratio
        A.PadIfNeeded(min_height=resize[0], min_width=resize[1], border_mode=cv2.BORDER_CONSTANT),
        A.Normalize(mean=[0.4627], std=[0.3013]),
        ToTensorV2(),
    ])

def overlay_mask(image_pil, mask, color=(0, 255, 0), alpha=0.4):
    """Overlay a probability mask (float32) onto image."""
    image = np.array(image_pil.convert("RGB"), dtype=np.float32) / 255.0
    mask = np.clip(mask, 0, 1)  # already a probability map

    # Create a solid color layer
    color_layer = np.ones_like(image)
    color_layer[:, :, 0] *= color[0] / 255.0
    color_layer[:, :, 1] *= color[1] / 255.0
    color_layer[:, :, 2] *= color[2] / 255.0

    # Blend mask onto image
    blended = image * (1 - mask[..., None] * alpha) + color_layer * (mask[..., None] * alpha)
    blended = np.clip(blended * 255, 0, 255).astype(np.uint8)

    return Image.fromarray(blended)

@torch.no_grad()
def process_image(image_pil: Image.Image, use_vit: bool = True):
    model.eval()
    transform = get_transform(resize)

    # Preprocess
    img_np = np.array(image_pil.convert("L"))  # grayscale
    aug = transform(image=img_np)
    img_tensor = aug["image"].unsqueeze(0).to(device)

    # Forward pass
    if use_vit:
        cls_logits, seg_logits = model_vit(img_tensor)
    else:
        cls_logits, seg_logits = model_novit(img_tensor)
        
    cls_idx = cls_logits.argmax(dim=1).item()
    class_name = LABEL_MAPS[cls_idx]

    # Segmentation: sigmoid since it’s multi-label
    seg_probs = torch.sigmoid(seg_logits)[0].cpu().numpy()  # (3, H, W)
    lung_mask = seg_probs[1]
    lesion_mask = seg_probs[2]

    # Resize back to original size
    orig_w, orig_h = image_pil.size
    lung_mask_resized = cv2.resize(lung_mask, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    lesion_mask_resized = cv2.resize(lesion_mask, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

    # --- Binary masks for area calculation ---
    lung_bin = (lung_mask_resized > threshold).astype(np.uint8)
    lesion_bin = (lesion_mask_resized > threshold).astype(np.uint8)

    lung_area = lung_bin.sum()
    lesion_area = lesion_bin.sum()
    infection_score = lesion_area / (lung_area + 1e-8) if lung_area > 0 else 0.0

    # --- Overlays (green lungs, red lesions) ---
    lung_overlay = overlay_mask(image_pil, lung_bin, color=(0, 255, 0))
    lesion_overlay = overlay_mask(image_pil, lesion_bin, color=(255, 0, 0))

    return f"Prediction: {class_name} | Severity: {infection_score}", lung_overlay, lesion_overlay

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=process_image,
    inputs=[
        gr.Image(type="pil", label="CXR Image"),
        gr.Radio(["CNN + ViT + Unet Model", "CNN + Unet Model"], label="Select Model", value="CNN + ViT + Unet Model")
    ],
    outputs=[
        gr.Label(label="Predictions"),
        gr.Image(label="Lung Mask"),
        gr.Image(label="Infection Mask"),
    ],
    title="CXR Disease and Severity Classifier",
    description="Upload a chest X-ray, select the model type, and get disease classification, lung/infection segmentation, and severity score.",
)

demo.launch(share=True)